In [ ]:
import os
os.chdir('..')

In [ ]:
import cv2
import numpy as np
import pandas as pd
import gymnasium as gym
import minari
from tqdm import tqdm
from minari.data_collector.episode_buffer import EpisodeBuffer

In [ ]:
DATA_DIR = '.cache/atari'
SCREENS_DIR = os.path.join(DATA_DIR, 'screens/revenge')
TRAJS_DIR = os.path.join(DATA_DIR, 'trajectories/revenge')

In [ ]:
def preprocess_frame(img_path):
    """Preprocessing individual frames in line with the method used by Mnih et al. [2015]."""
    # Read image data from PNG files
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read {img_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Extract luminance from frames to convert to greyscale
    weights = np.array([0.2125, 0.7154, 0.0721])
    gray = np.sum(np.multiply(img_rgb, weights), axis=-1).astype(np.uint8)

    # Resize frames to 84 x 84
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
def create_stacked_observations(frames, num_stack=4):
    """Creates stacked observations from a list of frames."""
    if len(frames) == 0:
        return np.array([])
        
    stacked_obs = []
    stack = [frames[0]] * num_stack
    
    for frame in frames:
        stack.pop(0)
        stack.append(frame)        
        stacked_obs.append(np.stack(stack, axis=0))
        
    return np.array(stacked_obs, dtype=np.uint8)

In [ ]:
# Creates local Minari dataset for Montezuma's Revenge

dataset_id = 'atari/montezuma-revenge-v0'

obs_space = gym.spaces.Box(low=0, high=255, shape=(4, 84, 84), dtype=np.uint8)
action_space = gym.spaces.Discrete(18)

dataset = minari.create_dataset_from_buffers(
    dataset_id=dataset_id,
    buffer=[],
    observation_space=obs_space,
    action_space=action_space
)

In [ ]:
# Writes Montezuma's Revenge data to the Minari dataset

traj_files = [f for f in os.listdir(TRAJS_DIR) if f.endswith('.txt')]

current_buffers = []
FLUSH_CHUNK_SIZE = 20

for i, traj_file in enumerate(tqdm(traj_files)):
    traj_id = traj_file.split('.')[0]
    traj_path = os.path.join(TRAJS_DIR, traj_file)
    screens_path = os.path.join(SCREENS_DIR, traj_id)

    if not os.path.exists(screens_path):
        continue

    with open(traj_path, 'r') as f:
        lines = f.readlines()

    header_idx = 0
    for idx, line in enumerate(lines):
        if line.startswith('frame'):
            header_idx = idx
            break

    df = pd.read_csv(traj_path, skiprows=header_idx, skipinitialspace=True)
    df.columns = [col.strip() for col in df.columns]

    actions = df['action'].values.astype(np.int32)[:-1]
    rewards = df['reward'].values.astype(np.float32)[:-1]
    terminals = df['terminal'].values.astype(bool)[:-1]
    frames_indices = df['frame'].values
    
    truncations = np.zeros_like(terminals)

    frames = []
    for frame_idx in frames_indices:
        img_path = os.path.join(screens_path, f"{frame_idx}.png")
        frames.append(preprocess_frame(img_path))

    stacked_obs = create_stacked_observations(frames, num_stack=4)

    ep_buffer = EpisodeBuffer(
        observations=stacked_obs,
        actions=actions,
        rewards=rewards,
        terminations=terminals,
        truncations=truncations,
        infos={}
    )

    current_buffers.append(ep_buffer)

    if len(current_buffers) >= FLUSH_CHUNK_SIZE:
        dataset.update_dataset_from_buffer(current_buffers)
        current_buffers = []

if current_buffers:
    dataset.update_dataset_from_buffer(current_buffers)